<a href="https://colab.research.google.com/github/mrishab/COMP381_FinalProject/blob/main/ClusteringExperiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Installation and set up
!pip install kagglehub[pandas-datasets]


In [17]:
# Imports
import kagglehub
from kagglehub import KaggleDatasetAdapter

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import classification_report, accuracy_score
from sklearn.cluster import KMeans
import math

In [18]:
# Set the path to the file you'd like to load
file_path = "cleaned_canada.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "yuliiabulana/canada-housing",
  file_path,
)

df.head()

Using Colab cache for faster access to the 'canada-housing' dataset.


,City,Province,Latitude,Longitude,Price,Bedrooms,Bathrooms,Acreage,Property Type,Square Footage,...,Exterior,Fireplace,Heating,Flooring,Roof,Waterfront,Sewer,Pool,Garden,Balcony
0,Revelstoke,BC,50.976585,-118.173149,839000.0,3.0,2.0,0.00,Condo,891.0,...,NaN,No,heat pump,carpet,NaN,No,municipal,No,No,No
1,Boswell,BC,49.471870,-116.770195,1150000.0,3.0,2.0,0.32,Single Family,1881.0,...,NaN,No,heat pump,NaN,NaN,No,septic,No,No,No
2,West Kelowna,BC,49.825230,-119.603253,149000.0,2.0,1.0,0.00,Single Family,912.0,...,Metal,No,NaN,laminate,tar,No,municipal,No,No,No
3,Kelowna,BC,49.821860,-119.480143,1298000.0,5.0,4.0,0.69,Single Family,4374.0,...,NaN,Yes,forced air,NaN,NaN,No,municipal,No,No,No
4,Maple Ridge,BC,49.221673,-122.596637,759900.0,3.0,2.0,0.00,Condo,1254.0,...,NaN,No,radiant,NaN,NaN,No,none,No,No,No


In [19]:
# Removing the outliers in dataset using the IQR of 3

main_features = ['Square Footage', 'Price']

q1 = df[main_features].quantile(0.25)
q3 = df[main_features].quantile(0.75)
iqr = q3 - q1

lower_threshold = q1 - 3 * iqr
upper_threshold = q3 + 3 * iqr

mask = pd.Series(True, index=df.index)  # start with all True

for col in main_features:
    mask &= (df[col] >= lower_threshold[col]) & (df[col] <= upper_threshold[col])

# Apply the mask safely
df_clean = df[mask]

df_clean.describe()

,Latitude,Longitude,Price,Bedrooms,Bathrooms,Acreage,Square Footage
count,42973.000000,42973.000000,4.297300e+04,42973.000000,42973.000000,42973.000000,42973.000000
mean,49.014079,-106.096160,8.590907e+05,3.125404,2.407768,2.278072,1636.263305
std,2.701268,22.419122,6.770967e+05,1.516894,1.226185,81.802771,939.154101
min,42.045940,-135.856018,5.000000e+04,0.000000,0.000000,0.000000,140.000000
25%,48.452283,-122.861128,3.980000e+05,2.000000,2.000000,0.000000,960.000000
50%,49.215293,-117.314748,6.590000e+05,3.000000,2.000000,0.000000,1377.000000
75%,50.058185,-101.889040,1.099900e+06,4.000000,3.000000,0.170000,2080.000000
max,65.281488,-52.668600,3.600000e+06,18.000000,14.000000,8600.000000,5866.000000


In [20]:
# Features Definitions
target = 'Price'

categorical_cols = [
    'City',
    'Province',
    'Property Type',
    'Basement',
    'Exterior',
    'Heating',
    'Flooring',
    'Roof',
    'Sewer'
]

numeric_cols = [
    'Bedrooms',
    'Bathrooms',
    'Square Footage',
    'Acreage',
    'Latitude',
    'Longitude'
]

binary_cols = [
    'Garage',
    'Parking',
    'Fireplace',
    'Waterfront',
    'Pool',
    'Garden',
    'Balcony'
]

binary_map = {'Yes': 1, 'No': 0}

In [21]:
# Hydrating with Enhanced Columns

def get_hydrated_df(df):
  # Replacing 'Yes'/'No' with 1/0
  for col in binary_cols:
      if col in df.columns:
          df[col] = df[col].map(binary_map)

  ## One-hot encoding
  df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

  ## Interaction Terms
  df_encoded['sqft_per_bedroom'] = df_encoded['Square Footage'] / (df_encoded['Bedrooms'] + 1)
  df_encoded['bath_bed_ratio'] = df_encoded['Bathrooms'] / (df_encoded['Bedrooms'] + 1)

  df_encoded['sqft_x_bedrooms'] = df_encoded['Square Footage'] * df_encoded['Bedrooms']
  df_encoded['sqft_x_bathrooms'] = df_encoded['Square Footage'] * df_encoded['Bathrooms']

  df_encoded['luxury_score'] = (
      df_encoded['Fireplace'] + df_encoded['Waterfront'] +
      df_encoded['Pool'] + df_encoded['Garden'] + df_encoded['Balcony']
  )

  ## Log transforms
  df_encoded['log_sqft'] = np.log1p(df_encoded['Square Footage'])
  df_encoded['log_acreage'] = np.log1p(df_encoded['Acreage'])


  return df_encoded


df_encoded = get_hydrated_df(df_clean.copy())
df_encoded.head()

,Latitude,Longitude,Price,Bedrooms,Bathrooms,Acreage,Square Footage,Garage,Parking,Fireplace,...,Sewer_none,Sewer_private,Sewer_septic,sqft_per_bedroom,bath_bed_ratio,sqft_x_bedrooms,sqft_x_bathrooms,luxury_score,log_sqft,log_acreage
0,50.976585,-118.173149,839000.0,3.0,2.0,0.00,891.0,1,1,0,...,False,False,False,222.75,0.500000,2673.0,1782.0,0,6.793466,0.000000
1,49.471870,-116.770195,1150000.0,3.0,2.0,0.32,1881.0,1,1,0,...,False,False,True,470.25,0.500000,5643.0,3762.0,0,7.540090,0.277632
2,49.825230,-119.603253,149000.0,2.0,1.0,0.00,912.0,0,0,0,...,False,False,False,304.00,0.333333,1824.0,912.0,0,6.816736,0.000000
3,49.821860,-119.480143,1298000.0,5.0,4.0,0.69,4374.0,1,0,1,...,False,False,False,729.00,0.666667,21870.0,17496.0,1,8.383662,0.524729
4,49.221673,-122.596637,759900.0,3.0,2.0,0.00,1254.0,1,1,0,...,True,False,False,313.50,0.500000,3762.0,2508.0,0,7.134891,0.000000


In [22]:
# Scale
scaler = StandardScaler()

df_scaled = pd.DataFrame(scaler.fit_transform(df_encoded), columns=df_encoded.columns)

X_scaled = df_scaled.drop(columns=[target])
y = df_encoded[target]


X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [23]:
feature_sets = {
    "baseline": [
        'Bedrooms', 'Bathrooms', 'Square Footage', 'Acreage'
    ],

    "ratios": [
        'sqft_per_bedroom', 'bath_bed_ratio', 'log_sqft'
    ],

    "interactions": [
        'sqft_x_bedrooms', 'sqft_x_bathrooms', 'luxury_score'
    ],

    "amenities": [
        'Garden', 'Fireplace', 'Pool', 'Waterfront', 'Balcony'
    ],

    "location": [
        'Latitude', 'Longitude'
    ],
}

# Adding categorical columns
encoded_cat_cols = [col for col in X.columns if any(c in col for c in categorical_cols)]
feature_sets["categorical_full"] = encoded_cat_cols

feature_sets["full_model"] = list(X.columns)

In [11]:
chosen_features = list(set([]
                           + feature_sets["interactions"]
                           + feature_sets["categorical_full"]
                           + feature_sets["ratios"]
                           + feature_sets["amenities"]
                           + feature_sets["location"]
                           ))

model = LinearRegression()
model.fit(X_train[chosen_features], y_train)

preds = model.predict(X_test[chosen_features])
r2 = r2_score(y_test, preds)

print(f"R2 = {r2}")

R2 = 0.6957339147975038


In [31]:
cluster_features = [
    'Latitude', 'Longitude'
]

km = KMeans(n_clusters=2, n_init='auto')
km.fit(df_scaled[cluster_features])

print("Cluster Labels = " + str(set(km.labels_)))
df_scaled['cluster'] = km.labels_

df_scaled.head()

Cluster Labels = {np.int32(0), np.int32(1)}


,Latitude,Longitude,Price,Bedrooms,Bathrooms,Acreage,Square Footage,Garage,Parking,Fireplace,...,Sewer_private,Sewer_septic,sqft_per_bedroom,bath_bed_ratio,sqft_x_bedrooms,sqft_x_bathrooms,luxury_score,log_sqft,log_acreage,cluster
0,0.726521,-0.538698,-0.029672,-0.082673,-0.332554,-0.027849,-0.793557,0.659194,1.170195,-0.814590,...,-0.048777,-0.323712,-0.945222,-0.415309,-0.536719,-0.555642,-0.784700,-0.856084,-0.373787,1
1,0.169475,-0.476119,0.429647,-0.082673,-0.332554,-0.023937,0.260596,0.659194,1.170195,-0.814590,...,-0.048777,3.089164,0.434995,-0.415309,-0.076915,-0.192073,-0.784700,0.530744,0.103486,1
2,0.300289,-0.602488,-1.048741,-0.741922,-1.148101,-0.027849,-0.771196,-1.517003,-0.854559,-0.814590,...,-0.048777,-0.323712,-0.492120,-1.246268,-0.668158,-0.715391,-0.784700,-0.812861,-0.373787,1
3,0.299041,-0.596997,0.648230,1.235826,1.298540,-0.019414,2.915143,0.659194,-0.854559,1.227611,...,-0.048777,-0.323712,1.877949,0.415651,2.435289,2.329772,0.794068,2.097647,0.528268,1
4,0.076851,-0.736009,-0.146496,-0.082673,-0.332554,-0.027849,-0.407034,0.659194,1.170195,-0.814590,...,-0.048777,-0.323712,-0.439142,-0.415309,-0.368124,-0.422333,-0.784700,-0.221900,-0.373787,1


In [32]:
for cluster_id in set(km.labels_):
  cluster_data = df_scaled[df_scaled['cluster'] == cluster_id]

  X_scaled = cluster_data.drop(columns=[target])
  y = cluster_data[target]


  X_train, X_test, y_train, y_test = train_test_split(
      X_scaled, y, test_size=0.2, random_state=42
  )

  cluster_model = LinearRegression()
  cluster_model.fit(X_train[chosen_features], y_train)

  cluster_preds = cluster_model.predict(X_test[chosen_features])
  r2 = r2_score(y_test, cluster_preds)

  print(f"CluterID: {cluster_id} : R2 = {r2}")

CluterID: 0 : R2 = 0.7111160929846496
CluterID: 1 : R2 = 0.7472029941041642


0 cluster = 0.69
2 cluster = 0.67 based on the price/square footage cluster
2 cluster = 0.69, 0.29, based on the custom feature clusters
2 cluster = 0.71, , based on the Latitude and Longitude